# Cross-account handoff probe

Attaches the **private** notebook output `khanhmay0304/tinyproto-fp-train-100client-s1` (owned by khanhmay0304) and reads it from **minhtrit06**. CPU only — this burns no GPU quota.

Answers the last open P2 item before the 100-client session-4 handoff.

In [ ]:
# Cross-account private-notebook-output handoff probe.
#
# The only open question from P2: can minhtrit06 READ the committed output of a PRIVATE notebook
# owned by khanhmay0304, attached as kernel_sources? Public datasets do not answer this. Session 4
# of the 100-client chain depends on it, and finding out at round 44 costs ~30 h of quota.
#
# This probe does what src/ckpt.py::import_previous actually does -- walk the tree and read the
# bytes -- rather than only checking that a path exists. A mounted-but-unreadable tree would pass
# an existence check and still break the resume.
import json, sys, hashlib
from pathlib import Path

SRC_SLUG = "tinyproto-fp-train-100client-s1"
inp = Path("/kaggle/input")
print("=== /kaggle/input ===")
if not inp.is_dir():
    sys.exit("FAIL: /kaggle/input does not exist at all")
for p in sorted(inp.iterdir()):
    print(" ", p.name)

roots = [p for p in inp.rglob("*") if p.is_dir() and p.parent.name == "runs"]
print("\n=== run directories found under /kaggle/input ===")
for r in roots:
    print(" ", r)
if not roots:
    sys.exit("FAIL: the attached kernel output is not readable from this account -- no runs/ tree. "
             "Session 4 must use a checkpoint DATASET instead of kernel_sources.")

run = roots[0]
cfg = json.loads((run / "config.json").read_text())
print(f"\nrun_name       {cfg['run_name']}")
print(f"scenario       {cfg['scenario']}  clients={cfg['n_clients']}  rounds={cfg['rounds']}")
print(f"mu_value       {cfg['mu_value']!r}")
print(f"fingerprint    {cfg['fingerprint']}")

done = sorted((run / "complete").glob("round_*.done"))
last = int(done[-1].stem.split("_")[-1]) if done else 0
print(f"complete       {len(done)} rounds, last = {last}")

# Read real bytes from the biggest artifact class, not just stat() it.
wts = sorted((run / "weights").glob("round_*.pt"))
print(f"weights        {len(wts)} files")
tot = 0
for p in wts:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk); tot += len(chunk)
print(f"weights bytes  {tot/2**30:.2f} GiB fully read, no I/O error")

import torch
w = torch.load(wts[-1], map_location="cpu", weights_only=False)
n = sum(v.numel() for v in w.values()) if isinstance(w, dict) else -1
print(f"torch.load     {wts[-1].name} OK, {n:,} tensor elements")

res = sorted((run / "resume").glob("*.pt"))
print(f"resume         {[p.name for p in res]}")
for p in res:
    p.read_bytes()
print("resume bytes   read OK")

print("\nPASS: a private notebook output owned by khanhmay0304 is fully readable from "
      "minhtrit06 via kernel_sources. The 100-client s3 -> s4 handoff can use kernel_sources.")